#👉 Добро пожаловать в конспект ученика!
Представься, пожалуйста:


## Урок 11 & 12. Архитектура RAG. Создание RAG-ассистента

Тут тебя ждут практические задания для 10 и 11 урока курса летней школы МШП по Векторизации и построению RAG систем.

---

Автор курса: Логинов Дмитрий Владимирович. Преподаватель и методист МШП, исследователь в области машинного обучения.



## 📋 Общее задание

**Цель:** Создать RAG-ассистента для школы, который умеет отвечать на вопросы на основе документов.

**Что нужно сделать:**
1. Подготовить документы о школе
2. Написать код для RAG-системы
3. Протестировать ассистента
4. Улучшить качество ответов

**Время на выполнение:** 100 минут

## 🛠️ Шаг 1. Установка библиотек (5 минут)

In [ ]:
# Установите необходимые библиотеки
!pip install -q faiss-cpu
!pip install -q sentence-transformers
!pip install langchain_community
!pip install PyPDF
!pip install -q requests
!pip install -q openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 347.3/347.3 kB 18.8 MB/s eta 0:00:00


## 📄 Шаг 2. Создание документов (10 минут)

**Задание:** Создайте 3-5 текстовых файлов с информацией о вашей школе.

In [ ]:
# Создаем файлы с информацией
# Заполните информацией о вашей школе

# Файл 1: Общая информация
school_info = """
Название школы: [Впишите название]
Год основания: [Впишите год]
Количество учеников: [Впишите число]
Количество классов: [Впишите число]

Директор: [Впишите ФИО]
Завуч: [Впишите ФИО]
"""

with open("school_info.txt", "w", encoding="utf-8") as f:
    f.write(school_info)

# Файл 2: Расписание
schedule = """
Расписание звонков:
1 урок: 8:30 - 9:15
2 урок: 9:25 - 10:10
3 урок: 10:20 - 11:05
4 урок: 11:20 - 12:05
5 урок: 12:20 - 13:05
6 урок: 13:15 - 14:00

Перемены:
После 1 урока - 10 минут
После 2 урока - 10 минут
После 3 урока - 15 минут
После 4 урока - 10 минут
После 5 урока - 10 минут
"""

with open("schedule.txt", "w", encoding="utf-8") as f:
    f.write(schedule)

# Файл 3: Правила
rules = """
Правила школы:
1. Приходить за 15 минут до начала занятий
2. Иметь сменную обувь
3. Телефоны сдавать в специальный ящик
4. Не опаздывать на уроки
5. Уважать учителей и одноклассников
"""

with open("rules.txt", "w", encoding="utf-8") as f:
    f.write(rules)

# Файл 4: Столовая
canteen = """
Столовая работает с 8:00 до 15:00

Меню на неделю:
Понедельник: Суп куриный, котлета с картофелем, компот
Вторник: Борщ, рыба с рисом, чай
Среда: Суп грибной, мясо с гречкой, сок
Четверг: Суп гороховый, сосиска с макаронами, какао
Пятница: Суп рисовый, плов, кисель

Стоимость обеда: 150 рублей
"""

with open("canteen.txt", "w", encoding="utf-8") as f:
    f.write(canteen)

# Файл 5: Кружки
clubs = """
Кружки и секции:

Понедельник:
- Шахматы (каб. 34) в 15:00

Вторник:
- Футбол (спортзал) в 14:00
- Рисование (каб. 21) в 15:30

Среда:
- Хор (актовый зал) в 15:00

Четверг:
- Английский клуб (каб. 56) в 14:00

Пятница:
- Каратэ (спортзал) в 15:00
"""

with open("clubs.txt", "w", encoding="utf-8") as f:
    f.write(clubs)

print("✅ Документы созданы!")

# Посмотрим, что создали
!ls -la *.txt

✅ Документы созданы!
-rw-r--r-- 1 root root 515 Jun 21 22:08 canteen.txt
-rw-r--r-- 1 root root 384 Jun 21 22:08 clubs.txt
-rw-r--r-- 1 root root 324 Jun 21 22:08 rules.txt
-rw-r--r-- 1 root root 410 Jun 21 22:08 schedule.txt
-rw-r--r-- 1 root root 326 Jun 21 22:08 school_info.txt


## 📚 Шаг 3. Импорт библиотек (3 минуты)

In [ ]:
# Импортируем все необходимые модули
import faiss
import numpy as np
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import os
import requests
import json
import openai

print("✅ Библиотеки импортированы!")

✅ Библиотеки импортированы!


## 🔑 Шаг 4. Настройка подключения к OpenRouter (5 минут)

In [ ]:
# Получите API ключ на https://openrouter.ai
OPENROUTER_API_KEY = ""  # Вставьте сюда ваш ключ

# Настраиваем подключение к OpenRouter через OpenAI
llm = openai.OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)


print("✅ Подключение к модели настроено!")

✅ Подключение к модели настроено!


## 📝 Шаг 5. Реализация RAG-пайплайна (60 минут)

**Задание:** Напишите код для всех этапов RAG. Каждый этап нужно реализовать самостоятельно.

In [ ]:
# ЭТАП 1: ЗАГРУЗКА ДОКУМЕНТОВ
# Ваш код:
# Используйте TextLoader для загрузки всех .txt файлов

# Нужно загрузить все файлы и сохранить в список documents

# ПОДСКАЗКА: используйте цикл или DirectoryLoader

# НАПИШИТЕ КОД ЗДЕСЬ:
documents = []
file_names = ["school_info.txt", "schedule.txt", "rules.txt", "canteen.txt", "clubs.txt"]

for file_name in file_names:
    loader = TextLoader(file_name, encoding="utf-8")
    docs = loader.load()
    documents.extend(docs)

In [ ]:
# ЭТАП 2: ЧАНКИНГ
# Ваш код:
# Создайте RecursiveCharacterTextSplitter с параметрами:
# chunk_size = 300
# chunk_overlap = 50

# НАПИШИТЕ КОД ЗДЕСЬ:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

chunks = splitter.split_documents(documents)
print(f"✅ Создано {len(chunks)} чанков")

In [ ]:
# ЭТАП 3: СОЗДАНИЕ ЭМБЕДДИНГОВ
# Ваш код:
# Используйте HuggingFaceEmbeddings с моделью "sentence-transformers/all-MiniLM-L6-v2"

# НАПИШИТЕ КОД ЗДЕСЬ:
model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("✅ Модель эмбеддингов создана")

In [ ]:
# ЭТАП 4: СОЗДАНИЕ ВЕКТОРНОЙ БАЗЫ
# Ваш код:
# Используйте FAISS.from_documents()

# НАПИШИТЕ КОД ЗДЕСЬ:


print("✅ Векторная база FAISS создана")

In [ ]:
# ЭТАП 5: ФУНКЦИЯ ПОИСКА
# Ваш код:
# Создайте функцию search(question), которая:
# 1. Ищет похожие документы (k=3)
# 2. Возвращает список найденных документов

# НАПИШИТЕ КОД ЗДЕСЬ:
def search(question, k=3):
   pass

print("✅ Функция поиска создана")

In [ ]:
# ЭТАП 6: СОЗДАНИЕ ПРОМПТА
# Ваш код:
# Используйте PromptTemplate для создания шаблона промпта

# НАПИШИТЕ КОД ЗДЕСЬ:
prompt_template = """
Ты помощник школы. Используй информацию из контекста для ответа.
Если ответа нет в контексте, скажи "Информация не найдена".

Контекст:
{context}

Вопрос:
{question}

Ответ:
"""

prompt =

print("✅ Промпт создан")


In [ ]:
# ЭТАП 7: ФУНКЦИЯ ОТВЕТА
# Ваш код:
# Создайте функцию answer(question), которая:
# 1. Ищет документы через search()
# 2. Формирует промпт
# 3. Отправляет запрос к LLM
# 4. Возвращает ответ

# НАПИШИТЕ КОД ЗДЕСЬ:
def answer(question):
    """Отвечает на вопрос с использованием RAG"""
    # Ищем документы
    docs = search(question)

    # Формируем контекст
    context = "\n\n".join([doc.page_content for doc in docs])

    # Формируем промпт
    formatted_prompt = prompt.format(context=context, question=question)

    # Отправляем запрос к LLM
    response = llm.invoke(formatted_prompt)

    return response.content

print("✅ Функция ответа создана")
print("\n🎉 RAG-пайплайн готов!")

## 🧪 Шаг 6. Проверка работы (15 минут)

**Задание:** Протестируйте ассистента на разных вопросах.

In [ ]:
# Примеры вопросов
test_questions = [
    "Как называется наша школа?",
    "Сколько уроков в день?",
    "Когда начинается первый урок?",
    "Что можно купить в столовой во вторник?",
    "Какие кружки есть в школе?",
    "Кто директор школы?",
    "Какие правила в школе?"
]

# Проверяем работу
print("🔍 Тестирование RAG-ассистента\n")
print("=" * 60)

for q in test_questions:
    print(f"\n❓ Вопрос: {q}")
    try:
        response = answer(q)
        print(f"💡 Ответ: {response}")
    except Exception as e:
        print(f"❌ Ошибка: {e}")
    print("-" * 60)

## 🎨 Шаг 7. Улучшение промпта (15 минут)

**Задание:** Улучшите промпт, чтобы ответы были лучше.

In [ ]:
# Создайте улучшенный промпт
# Включите:
# 1. Четкую инструкцию
# 2. Правила ответа
# 3. Формат вывода

# НАПИШИТЕ УЛУЧШЕННЫЙ ПРОМПТ ЗДЕСЬ:
improved_prompt_template = """
Ты умный и внимательный помощник школы. Твоя задача - давать точные и полезные ответы.

Правила ответа:
1. Используй ТОЛЬКО информацию из контекста
2. Если информации нет, честно скажи об этом
3. Отвечай подробно и понятно
4. Будь вежливым и дружелюбным

Контекст:
{context}

Вопрос пользователя:
{question}

Подробный ответ:
"""

improved_prompt =

# Создайте функцию improved_answer()
# НАПИШИТЕ КОД ЗДЕСЬ:
def improved_answer(question):
    """Отвечает на вопрос с использованием улучшенного промпта"""
    # Ищем документы
    docs = search(question)

    # Формируем контекст
    context = "\n\n".join([doc.page_content for doc in docs])

    # Формируем промпт
    formatted_prompt = improved_prompt.format(context=context, question=question)

    # Отправляем запрос к LLM
    response = llm.invoke(formatted_prompt)

    return response.content

print("✅ Улучшенная функция ответа создана!")

# Проверьте улучшенные ответы
print("\n🔍 Тестирование улучшенного ассистента\n")
print("=" * 60)

test_question = "Расскажи подробно о кружках в школе"
print(f"\n❓ Вопрос: {test_question}")
try:
    response = improved_answer(test_question)
    print(f"💡 Ответ: {response}")
except Exception as e:
    print(f"❌ Ошибка: {e}")
print("-" * 60)

## 📊 Шаг 8. Чат с ассистентом (15 минут)

**Задание:** Создайте простой чат-интерфейс для общения с ассистентом.

In [ ]:
# Создайте функцию для чата
def chat():
    """Запускает диалог с ассистентом"""
    print("🤖 Добро пожаловать в чат с RAG-ассистентом!")
    print("Введите 'выход' или 'exit' для завершения\n")

    while True:
        # Получаем вопрос от пользователя
        question = input("👤 Вы: ")

        # Проверяем выход
        if question.lower() in ["выход", "exit", "quit"]:
            print("👋 До свидания!")
            break

        # Получаем ответ
        try:
            answer_text = improved_answer(question)
            print(f"🤖 Ассистент: {answer_text}\n")
        except Exception as e:
            print(f"❌ Ошибка: {e}\n")

# Запускаем чат
chat()

## 📝 Шаг 9. Сохранение и загрузка базы (2 минуты)

**Задание:** Научитесь сохранять и загружать векторную базу.

In [ ]:
# Сохраняем базу
vectorstore.save_local("school_faiss_index")
print("✅ База сохранена!")

# Загружаем базу
loaded_vectorstore = FAISS.load_local(
    "school_faiss_index",
    embeddings,
    allow_dangerous_deserialization=True
)
print("✅ База загружена!")

## ✅ Чек-лист выполнения

Отметьте, что вы сделали:

- [ ] Установили библиотеки
- [ ] Создали документы о школе (3-5 файлов)
- [ ] Импортировали все модули
- [ ] Настроили подключение к OpenRouter
- [ ] Реализовали загрузку документов
- [ ] Реализовали чанкинг
- [ ] Создали эмбеддинги
- [ ] Создали векторную базу FAISS
- [ ] Написали функцию поиска
- [ ] Создали промпт
- [ ] Написали функцию ответа
- [ ] Проверили работу на вопросах
- [ ] Улучшили промпт
- [ ] Создали чат-интерфейс
- [ ] Сохранили векторную базу

## 🎯 Критерии оценки

| Критерий | Баллы |
|----------|-------|
| Все этапы RAG реализованы правильно | 30 |
| Код работает без ошибок | 20 |
| Ассистент правильно отвечает на 5+ вопросов | 20 |
| Создан улучшенный промпт | 10 |
| Создан чат-интерфейс | 10 |
| Код оформлен аккуратно (комментарии, отступы) | 10 |
| **Итого:** | **100** |

## 🚀 Бонусное задание (если осталось время)

Создайте веб-интерфейс для ассистента с помощью Gradio:

In [ ]:
!pip install gradio -q
import gradio as gr

def respond(message, history):
    return improved_answer(message)

gr.ChatInterface(
    fn=respond,
    title="Школьный RAG-ассистент",
    description="Задайте любой вопрос о школе!"
).launch()

## 🎉 Поздравляю!

Вы создали своего первого RAG-ассистента! Теперь вы знаете:
- Как загружать и обрабатывать документы
- Как разбивать текст на чанки
- Как создавать эмбеддинги
- Как использовать векторную базу данных
- Как настраивать промпты
- Как получать умные ответы на вопросы